In [53]:
#import libraries
import sqlite3
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [54]:
#step:1 create database
conn=sqlite3.connect("student_database.db")

#load csv file
df=pd.read_csv(r"C:\Users\USER\Downloads\student_scores.csv")

#save dataset in database
df.to_sql("Student",conn,if_exists="replace",index=False)
print("data saved succesfully")

#load dataset from db
df=pd.read_sql("SELECT *FROM Student",conn)
print(df.head())

data saved succesfully
   hours_studied  sleep_hours  previous_score  final_score
0              1            6              45           48
1              2            7              50           55
2              3            6              52           58
3              4            8              60           65
4              5            7              65           72


In [55]:
#step:2 EDA
print(df.shape)
print("\n",df.dtypes)
print("\n",df.isna().sum())
df.describe()

(50, 4)

 hours_studied     int64
sleep_hours       int64
previous_score    int64
final_score       int64
dtype: object

 hours_studied     0
sleep_hours       0
previous_score    0
final_score       0
dtype: int64


,hours_studied,sleep_hours,previous_score,final_score
count,50.000000,50.000000,50.000000,50.000000
mean,4.640000,6.720000,62.160000,68.800000
std,2.173683,0.904411,13.993235,14.824139
min,1.000000,5.000000,38.000000,42.000000
25%,3.000000,6.000000,50.250000,57.250000
50%,5.000000,7.000000,62.500000,69.500000
75%,6.000000,7.000000,72.750000,80.750000
max,8.000000,8.000000,90.000000,97.000000


In [56]:

#features and target
x=df.iloc[:,:-1]
y=df["final_score"]
#train-test-split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [57]:
#step3:preprocessing
#scalling is required because previous_score values are more compare to study and sleep hours
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [58]:
#step4:train model
model=LinearRegression()
model.fit(x_train,y_train)

LinearRegression()

In [59]:
#step5:save model
import joblib
joblib.dump(model,"student_model.pkl")
joblib.dump(scaler,"student_scaler.pkl")

['student_scaler.pkl']

In [60]:
#evalaution
y_pred_train=model.predict(x_train)
y_Pred_test=model.predict(x_test)
train_acc=r2_score(y_train,y_pred_train)
test_acc=r2_score(y_test,y_Pred_test)
print("Train accuracy:",train_acc)
print("Test accuracy:",test_acc)
print("MAE:",mean_absolute_error(y_test,y_Pred_test))
print("MSE:",mean_squared_error(y_test,y_Pred_test))
RMSE=np.sqrt(mean_squared_error(y_test,y_Pred_test))
print("RMSE:",RMSE)

Train accuracy: 0.9975166942347607
Test accuracy: 0.994655968881716
MAE: 0.4600357165985812
MSE: 0.4320114756020799
RMSE: 0.6572757987345038


In [61]:
#step6: build fastapi